# 04 带 KV Cache 的 Causal MHA 白板编程

**题目:** 在 Causal MHA 基础上支持 KV Cache

- `CausalMHAWithKVCache(nn.Module)`
- `forward(self, x, kv_cache=None)`
- **Prefill**: `kv_cache=None`, `x` 是完整 prompt `(B, T, d_model)`
- **Decode**: `kv_cache=(K_cache, V_cache)`, `x` 是新 token `(B, 1, d_model)`
- 返回 `(output, new_kv_cache)`
- `new_kv_cache = (K_all, V_all)` 形状 `(B, num_heads, total_seq_len, d_k)`

**思路:**
1. 对 x 算 Q, K, V 并拆头
2. 如果有 cache，把新 K/V 拼到 cache 后面（dim=2，即 seq_len 维度）
3. Q 用新的，K/V 用拼接后的完整版
4. prefill 时需要 causal mask，decode 时 Q 只有 1 个 token 不需要 mask

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)
print("ready")

In [ ]:
# ✏️ 在这里写你的代码

class CausalMHAWithKVCache(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        pass  # 👈 写在这里
    
    def forward(self, x, kv_cache=None):
        """
        x: (B, T, d_model)  T=seq_len(prefill) 或 T=1(decode)
        kv_cache: None 或 (K_cache, V_cache)
           K_cache: (B, num_heads, cached_len, d_k)
           V_cache: (B, num_heads, cached_len, d_k)
        
        返回: (output, new_kv_cache)
           output: (B, T, d_model)
           new_kv_cache: (K_all, V_all)
        """
        pass  # 👈 写在这里

In [ ]:
# ✅ 测试
torch.manual_seed(0)
model = CausalMHAWithKVCache(d_model=32, num_heads=4)
model.eval()

# 构造输入: 6 个 token
x_full = torch.randn(1, 6, 32)

# --- 方法 A: 一次性算（无 cache，当 baseline）---
with torch.no_grad():
    out_full, _ = model(x_full, kv_cache=None)

# --- 方法 B: 先 prefill 前 4 个，再逐个 decode 后 2 个 ---
with torch.no_grad():
    out_prefill, cache = model(x_full[:, :4, :], kv_cache=None)
    out_d1, cache = model(x_full[:, 4:5, :], kv_cache=cache)
    out_d2, cache = model(x_full[:, 5:6, :], kv_cache=cache)

out_incremental = torch.cat([out_prefill, out_d1, out_d2], dim=1)

# 验证: 两种方法输出一致
diff = (out_full - out_incremental).abs().max().item()
assert diff < 1e-4, f"两种方法输出不一致! 最大差异: {diff:.6f}"

# 验证 cache shape
K_cache, V_cache = cache
assert K_cache.shape == (1, 4, 6, 8), f"K cache shape 应该是 (1,4,6,8), 你的是 {K_cache.shape}"

print("✅ 通过!")
print(f"   全量 vs 增量 最大差异: {diff:.2e}")
print(f"   最终 cache shape: K={K_cache.shape}, V={V_cache.shape}")